# 📈 Introducción al modelo ARIMA


El modelo ARIMA (AutoRegresivo Integrado de Media Móvil) es uno de los más utilizados en series de tiempo para modelar y predecir valores futuros. Combina tres ideas fundamentales:

- **AR (AutoRegresivo)**: la serie depende de sus propios rezagos.
- **I (Integrado)**: se refiere al número de diferenciaciones necesarias para hacer estacionaria la serie.
- **MA (Media Móvil)**: la serie depende de errores pasados.

### 1️⃣ 🧮 Parámetros del modelo ARIMA (p, d, q)


- **p**: número de términos autorregresivos (AR).
- **d**: número de veces que la serie debe diferenciarse para volverla estacionaria.
- **q**: número de términos de media móvil (MA).

El proceso de selección de estos parámetros se puede dividir en dos partes: determinar **d** (estacionariedad), y luego usar **ACF** y **PACF** para elegir **p** y **q**.

### 2️⃣ 📥 Carga de datos


In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv")
df.columns = ["Fecha", "Pasajeros"]
df["Fecha"] = pd.to_datetime(df["Fecha"])
df.set_index("Fecha", inplace=True)
df

### 3️⃣ 📊 Visualización de la serie original


In [ ]:
import matplotlib.pyplot as plt

df.plot(figsize=(10, 4), title="Número de pasajeros mensuales", ylabel="Pasajeros")
plt.grid()
plt.show()

### 4️⃣ ⚖️ Estacionariedad y determinación de 'd'


Primero, necesitamos asegurarnos de que la serie sea estacionaria. Esto significa que sus propiedades estadísticas (media, varianza) no cambian con el tiempo.

Usamos la **prueba de Dickey-Fuller aumentada (ADF)** para verificar esto. Si el valor p es mayor a 0.05, la serie **no es estacionaria** y debe diferenciarse.

In [ ]:
from statsmodels.tsa.stattools import adfuller

resultado_adf = adfuller(df["Pasajeros"])
print(f"Valor p: {resultado_adf[1]}")

In [ ]:
resultado_adf

Como el valor p es mayor que 0.05, la serie **no es estacionaria**. Aplicamos una diferenciación (d=1).

#### 🔁 Serie diferenciada y nueva prueba ADF


In [ ]:
df_diff = df.diff().dropna()
df_diff

In [ ]:
resultado_adf_diff = adfuller(df_diff["Pasajeros"])
print(f"Valor p tras una diferenciación: {resultado_adf_diff[1]}")

In [ ]:
resultado_adf_diff[1] < 0.05

In [ ]:
# Segunda diferencia
df_diff2 = df_diff.diff().dropna()
adfuller(df_diff2["Pasajeros"])[1] < 0.05

Ahora el valor p es menor que 0.05, por lo tanto, la serie es **estacionaria** después de una diferenciación: **$d = 1$ o $2$**.

### 5️⃣ 📉 ACF y PACF para determinar $p$ y $q$


La función de autocorrelación (ACF) y la autocorrelación parcial (PACF) nos ayudan a determinar los valores de **q** y **p** respectivamente.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
df_diff

In [ ]:
plot_acf(df_diff2, lags=20)
plt.show()

In [ ]:
plot_pacf(df_diff2, lags=20)
plt.show()

- Observamos los cortes significativos en los gráficos para estimar:
  - El número de rezagos donde **PACF** cae a cero: sugiere el valor de **p**.
  - El número de rezagos donde **ACF** cae a cero: sugiere el valor de **q**.